In [0]:
from pyspark.sql import functions as F

# Leer los registros válidos de Silver
df_silver = spark.table("aws_s3.ventas.silver_ventas")

# Calcular el importe de venta utilizando los valores limpios
df_ventas = (
    df_silver
    .withColumn(
        "venta_calculada_crc",
        F.round(F.col("total_esperado"), 2)
    )
    .withColumn(
        "fecha_venta",
        F.col("fecha_normalizada")
    )
)

# Revisar la estructura y los datos
df_ventas.printSchema()
display(df_ventas.limit(10))

In [0]:
# Esta tabla permitirá analizar la evolución de los ingresos y el volumen de ventas por día.
gold_diarias = (
    df_ventas
    .groupBy("fecha_venta")
    .agg(
        F.countDistinct("id_venta").alias("cantidad_ventas"),
        F.sum("cantidad_limpia").alias("unidades_vendidas"),
        F.round(
            F.sum("venta_calculada_crc"), 2
        ).alias("ingresos_crc"),
        F.round(
            F.avg("venta_calculada_crc"), 2
        ).alias("ticket_promedio_crc"),
        F.round(
            F.avg("descuento_limpio"), 2
        ).alias("descuento_promedio_pct")
    )
    .orderBy("fecha_venta")
)

gold_diarias.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("aws_s3.ventas.gold_ventas_diarias")

display(gold_diarias)

In [0]:
# Esta tabla ayudará a identificar los productos que generan mayores ingresos y volumen de ventas.
gold_producto = (
    df_ventas
    .groupBy("producto")
    .agg(
        F.countDistinct("id_venta").alias("cantidad_ventas"),
        F.sum("cantidad_limpia").alias("unidades_vendidas"),
        F.round(
            F.sum("venta_calculada_crc"), 2
        ).alias("ingresos_crc"),
        F.round(
            F.avg("venta_calculada_crc"), 2
        ).alias("ticket_promedio_crc")
    )
    .orderBy(F.desc("ingresos_crc"))
)

gold_producto.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("aws_s3.ventas.gold_ventas_producto")

display(gold_producto)

In [0]:
# Esta tabla permitirá comparar las ventas entre provincias.
gold_provincia = (
    df_ventas
    .groupBy("provincia")
    .agg(
        F.countDistinct("id_venta").alias("cantidad_ventas"),
        F.sum("cantidad_limpia").alias("unidades_vendidas"),
        F.round(
            F.sum("venta_calculada_crc"), 2
        ).alias("ingresos_crc"),
        F.round(
            F.avg("venta_calculada_crc"), 2
        ).alias("ticket_promedio_crc")
    )
    .orderBy(F.desc("ingresos_crc"))
)

gold_provincia.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("aws_s3.ventas.gold_ventas_provincia")

display(gold_provincia)

In [0]:
# Con esta tabla podrás estudiar qué métodos de pago y canales comerciales concentran los ingresos
gold_canales = (
    df_ventas
    .groupBy("metodo_pago", "canal_venta")
    .agg(
        F.countDistinct("id_venta").alias("cantidad_ventas"),
        F.round(
            F.sum("venta_calculada_crc"), 2
        ).alias("ingresos_crc"),
        F.round(
            F.avg("venta_calculada_crc"), 2
        ).alias("ticket_promedio_crc")
    )
    .orderBy(F.desc("ingresos_crc"))
)

gold_canales.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("aws_s3.ventas.gold_ventas_canales")

display(gold_canales)